s: {"date":"2026-09-30T15:46:32.315Z","msg_id":"48d1485e-e30f-47da-b1d2-1e2f91b63e6c","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#W6sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"run_main_with_args([\n    '--steps', '10000',\n    '--ctx', '128',\n    '--ent_warmup', '0'\n])\n#this one"}


In [13]:
run_main_with_args([
    '--steps', '10000',
    '--ctx', '128',
    '--ent_warmup', '0'
])
#this one

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cuda
step     0 | soft BPC 3.592 | hard BPC 3.591 | entropy 0.590 nats | 0s
step   250 | soft BPC 3.029 | hard BPC 3.998 | entropy 0.443 nats | 16s
step   500 | soft BPC 2.871 | hard BPC 4.047 | entropy 0.345 nats | 31s
step   750 | soft BPC 2.795 | hard BPC 3.907 | entropy 0.292 nats | 45s
step  1000 | soft BPC 2.739 | hard BPC 3.857 | entropy 0.254 nats | 60s
step  1250 | soft BPC 2.725 | hard BPC 3.934 | entropy 0.224 nats | 75s
step  1500 | soft BPC 2.698 | hard BPC 4.361 | entropy 0.201 nats | 92s
step  1750 | soft BPC 2.686 | hard BPC 4.459 | entropy 0.183 nats | 108s
step  2000 | soft BPC 2.682 | hard BPC 4.688 | entropy 0.169 nats | 122s
step  2250 | soft BPC 2.653 | hard BPC 4.883 | entropy 0.158 nats | 136s
step  2500 | soft BPC 2.637 | hard BPC 4.913 | entropy 0.148 nats | 151s
step  2750 | soft BPC 2.642 | hard BPC 5.092 | entropy 0.140 nats | 165s
step  3000 | soft BPC

s: {"date":"2026-09-30T16:31:17.741Z","msg_id":"e0c3bb71-592a-46df-beac-12a694ceb079","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#X20sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"run_main_with_args([\n    '--steps', '20000',\n    '--batch', '4',\n    '--ctx', '128',\n    '--eval_every', '500',\n    '--eval_seqs', '64',\n    '--eval_chunk_size', '4',\n    '--no_ckpt'\n])"}


---asdads
###next###


In [27]:
"""gf256_markov_lm.py  -  Lagrange Field Transformer, prototype v6
=================================================================
V6 changes:
  * CRANKS UP parameter entropy penalty (ent_max=2.0) to force 99.9% 
    deterministic S-boxes, preventing "Cascading Chaos" in the hard path.
  * Delays STE loss until step 10000 to avoid noisy gradients on broad distributions.
  * Drops LR by 5x when STE starts to gently align the readout tables.
"""
import argparse, math, os, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

import random
import numpy as np


def build_gf_tables():
    exp = [0] * 512
    log = [0] * 256
    x = 1
    for i in range(255):
        exp[i] = x
        log[x] = i
        x2 = x << 1
        if x2 & 0x100:
            x2 ^= 0x11B
        x = x2 ^ x
    for i in range(255, 512):
        exp[i] = exp[i - 255]
    return exp, log

EXP, LOG = build_gf_tables()

def gf_mul(a, b):
    if a == 0 or b == 0: return 0
    return EXP[LOG[a] + LOG[b]]

def fwht(x):
    shape = x.shape
    x = x.reshape(-1, 256)
    h = 1
    while h < 256:
        x = x.view(-1, 256 // (2 * h), 2, h)
        a, b = x[:, :, 0, :], x[:, :, 1, :]
        x = torch.stack((a + b, a - b), dim=2)
        h *= 2
    return x.reshape(shape)

def xor_convolve(dists):
    acc = None
    for d in dists:
        w = fwht(d)
        acc = w if acc is None else acc * w
    out = fwht(acc) / 256.0
    out = out.clamp_min(0.0)
    return out / out.sum(-1, keepdim=True).clamp_min(1e-12)

def ste_softmax(logits, ste):
    soft = F.softmax(logits, -1)
    if not ste:
        return soft
    hard = F.one_hot(soft.argmax(dim=-1), num_classes=logits.shape[-1]).to(logits.dtype)
    return hard + (soft - soft.detach())

def row_entropy(p):
    return -(p * (p + 1e-12).log()).sum(-1).mean()

class GFLayer(nn.Module):
    def __init__(self, d, fan_in, lags, coef_set, gen):
        super().__init__()
        self.d, self.fan_in, self.lags = d, fan_in, lags
        K = len(coef_set)

        ch = torch.randint(0, d, (d, fan_in), generator=gen)
        lg = torch.randint(0, len(lags), (d, fan_in), generator=gen)
        ch[:, 0] = torch.arange(d)
        lg[:, 0] = lags.index(0)

        self.register_buffer("ch", ch)
        self.register_buffer("lag", lg)
        self.register_buffer("lag_vals", torch.tensor(lags)[lg])

        self.coef_logits = nn.Parameter(0.1 * torch.randn(d, fan_in, K, generator=gen))
        with torch.no_grad():
            one_idx = coef_set.index(1)
            zero_idx = coef_set.index(0)
            self.coef_logits[:, 0, one_idx] += 6.0
            if fan_in > 1:
                self.coef_logits[:, 1:, zero_idx] += 4.0

        sbox = 0.1 * torch.randn(d, 256, 256, generator=gen)
        ar256 = torch.arange(256)
        sbox[torch.arange(d)[:, None], ar256[None, :], ar256[None, :]] += 8.0
        self.sbox_logits = nn.Parameter(sbox)
        self.ste = False

    def sbox(self, ste=False): return ste_softmax(self.sbox_logits, ste)
    def coefs(self, ste=False): return ste_softmax(self.coef_logits, ste)

    def forward(self, x, perm_mats):
        B, T, d, _ = x.shape
        max_lag = max(self.lags)
        pad = torch.zeros(B, max_lag, d, 256, device=x.device, dtype=x.dtype)
        pad[..., 0] = 1.0
        xp = torch.cat([pad, x], dim=1)

        M = torch.einsum("dfk,kij->dfij", self.coefs(self.ste), perm_mats)
        t = torch.arange(T, device=x.device)[:, None]
        parts = []
        for f in range(self.fan_in):
            tidx = t + max_lag - self.lag_vals[:, f][None, :]
            src = xp[:, tidx, self.ch[:, f][None, :]]
            parts.append(torch.einsum("btdi,dij->btdj", src, M[:, f]))
        
        z = xor_convolve(parts)
        return torch.einsum("btdi,dij->btdj", z, self.sbox(self.ste))


class GFMarkovLM(nn.Module):
    def __init__(self, d=32, layers=2, fan_in=3, lags=(0, 1, 2, 4, 8, 16, 32, 64),
                 n_coef=8, bigram_logp=None, seed=0):
        super().__init__()
        gen = torch.Generator().manual_seed(seed)
        self.d = d
        self.lags = list(lags)

        coef_set = [0, 1]
        while len(coef_set) < n_coef:
            c = int(torch.randint(2, 256, (1,), generator=gen))
            if c not in coef_set:
                coef_set.append(c)
        self.coef_set = coef_set

        pm = torch.zeros(len(coef_set), 256, 256)
        for k, c in enumerate(coef_set):
            for i in range(256):
                pm[k, i, gf_mul(c, i)] = 1.0
        self.register_buffer("perm_mats", pm)
        self.use_ckpt = True

        emb = 0.1 * torch.randn(256, d, 256, generator=gen)
        ar256 = torch.arange(256)
        emb[ar256, 0, ar256] += 10.0
        for j in range(1, d):
            emb[ar256, j, torch.randperm(256, generator=gen)] += 8.0
        self.emb_logits = nn.Parameter(emb)

        self.layers = nn.ModuleList([
            GFLayer(d, fan_in, self.lags, coef_set, gen) for _ in range(layers)
        ])

        self.readout = nn.ParameterList([
            nn.Parameter(torch.zeros(d, 256, 256)) for _ in range(layers + 1)
        ])
        if bigram_logp is not None:
            with torch.no_grad():
                self.readout[0][0] = bigram_logp

        self.readout_bias = nn.Parameter(torch.zeros(256))
        self.ste = False

    def set_ste(self, on):
        self.ste = on
        for L in self.layers:
            L.ste = on

    def forward(self, tokens):
        h = ste_softmax(self.emb_logits, self.ste)[tokens]
        logits = torch.einsum("btdi,div->btv", h, self.readout[0])
        for layer, R in zip(self.layers, self.readout[1:]):
            if self.training and self.use_ckpt:
                h = torch.utils.checkpoint.checkpoint(layer, h, self.perm_mats, use_reentrant=False)
            else:
                h = layer(h, self.perm_mats)
            logits = logits + torch.einsum("btdi,div->btv", h, R)
        return logits + self.readout_bias

    def regularisers(self):
        ent = row_entropy(F.softmax(self.emb_logits, -1))
        perm_pen = 0.0
        for L in self.layers:
            P = L.sbox(False)
            C = L.coefs(False)
            ent = ent + row_entropy(P) + row_entropy(C)
            perm_pen = perm_pen + ((P.sum(1) - 1.0) ** 2).mean()
        return ent / (1 + 2 * len(self.layers)), perm_pen / max(1, len(self.layers))

    @torch.no_grad()
    def export_hard(self):
        return {
            "emb": self.emb_logits.argmax(-1),
            "coef": [torch.tensor(self.coef_set, device=L.coef_logits.device)[L.coef_logits.argmax(-1)] for L in self.layers],
            "sbox": [L.sbox_logits.argmax(-1) for L in self.layers],
            "wiring": [(L.ch, L.lag) for L in self.layers],
            "readout": [R.detach() for R in self.readout],
            "bias": self.readout_bias.detach(),
        }

    @torch.no_grad()
    def hard_forward(self, tokens, hard):
        exp_t = torch.tensor(EXP)
        log_t = torch.tensor(LOG)
        lags = torch.tensor(self.lags)
        B, T = tokens.shape
        d = self.d
        h = hard["emb"][tokens]
        ar = torch.arange(d)
        logits = hard["readout"][0][ar, h].sum(2)

        for coef, sbox, (ch, lg), R in zip(hard["coef"], hard["sbox"], hard["wiring"], hard["readout"][1:]):
            acc = torch.zeros(B, T, d, dtype=torch.long)
            for f in range(coef.shape[1]):
                out = torch.zeros(B, T, d, dtype=torch.long)
                for j in range(d):
                    l = int(lags[lg[j, f]])
                    src = h[:, :, ch[j, f]]
                    if l:
                        src = torch.cat([torch.zeros(B, l, dtype=torch.long), src[:, :-l]], dim=1)
                    c = int(coef[j, f])
                    if c == 0:
                        prod = torch.zeros_like(src)
                    else:
                        prod = torch.where(src == 0, torch.zeros_like(src), exp_t[log_t[src] + LOG[c]])
                    out[:, :, j] = prod
                acc = acc ^ out
            h = sbox[ar, acc]
            logits = logits + R[ar, h].sum(2)
        return logits + hard["bias"]

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

def load_data(path):
    if path is None:
        path = "tinyshakespeare.txt"
        if not os.path.exists(path):
            print("downloading Tiny Shakespeare ...")
            urllib.request.urlretrieve(URL, path)
    raw = open(path, "rb").read()
    data = torch.tensor(list(raw), dtype=torch.long)
    n = int(0.9 * len(data))
    return data[:n], data[n:]

def bigram_logp(data):
    counts = torch.ones(256, 256)
    counts.index_put_((data[:-1], data[1:]), torch.ones(len(data) - 1), accumulate=True)
    return (counts / counts.sum(1, keepdim=True)).log()

def ngram_baselines(train, val):
    uni = torch.bincount(train, minlength=256).float() + 1
    uni = uni / uni.sum()
    c2 = torch.bincount(train[:-1] * 256 + train[1:], minlength=256**2).float().view(256, 256)
    c3 = torch.bincount(train[:-2] * 65536 + train[1:-1] * 256 + train[2:], minlength=256**3).float()
    a, b, c = val[:-2], val[1:-1], val[2:]
    p1 = uni[c]
    p2 = c2[b, c] / c2[b].sum(1).clamp_min(1)
    ab = c2[a, b]
    p3 = c3[a * 65536 + b * 256 + c] / ab.clamp_min(1)
    bi = -(0.95 * p2 + 0.05 * p1).log2().mean().item()
    tri = -(0.6 * p3 + 0.3 * p2 + 0.1 * p1).log2().mean().item()
    return bi, tri

def get_batch(data, B, T, device):
    ix = torch.randint(0, len(data) - T - 1, (B,))
    x = torch.stack([data[i:i + T] for i in ix])
    y = torch.stack([data[i + 1:i + T + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def evaluate(model, seqs, device, chunk=64):
    was_ste = model.ste
    model.eval()
    model.set_ste(False)
    x, y = seqs[:, :-1], seqs[:, 1:]
    soft_nll, hard_nll = 0.0, 0.0
    for k in range(0, len(x), chunk):
        xb, yb = x[k:k + chunk], y[k:k + chunk]
        lo = model(xb.to(device)).float().cpu()
        soft_nll += F.cross_entropy(lo.reshape(-1, 256), yb.reshape(-1), reduction="sum").item()
    
    hard = model.export_hard()
    hard = {k: ([t.cpu() for t in v] if isinstance(v, list) and not isinstance(v[0], tuple) else [(a.cpu(), b.cpu()) for a, b in v] if isinstance(v, list) else v.cpu()) for k, v in hard.items()}
    
    for k in range(0, len(x), chunk):
        xb, yb = x[k:k + chunk], y[k:k + chunk]
        lo = model.hard_forward(xb, hard)
        hard_nll += F.cross_entropy(lo.reshape(-1, 256), yb.reshape(-1), reduction="sum").item()
    
    n = y.numel() * math.log(2)
    model.set_ste(was_ste)
    model.train()
    return soft_nll / n, hard_nll / n

def main(args):

    # --- ADD THIS BLOCK TO FORCE REPRODUCIBILITY ---
    seed = args.seed  # Change this number to try different "locks"!
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
 
    train, val = load_data(args.data)
    bi, tri = ngram_baselines(train, val)
    print(f"baselines on full val: bigram {bi:.3f} BPC | trigram {tri:.3f} BPC")

    model = GFMarkovLM(d=args.d, layers=args.layers, fan_in=args.fan_in, bigram_logp=bigram_logp(train)).to(args.device)
    model.use_ckpt = not args.no_ckpt
    
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    n_par = sum(p.numel() for p in model.parameters())
    print(f"params {n_par/1e6:.2f}M | d={args.d} layers={args.layers} fan_in={args.fan_in} device={args.device}")

    L = args.ctx + 1
    all_val = val[: (len(val) // L) * L].view(-1, L)
    gsel = torch.Generator().manual_seed(123)
    quick_val = all_val[torch.randperm(len(all_val), generator=gsel)[:args.eval_seqs]]

    t0 = time.time()

    for step in range(args.steps + 1):
        if step % args.eval_every == 0 or step == args.steps:
            soft, hard = evaluate(model, quick_val, args.device, chunk=args.eval_chunk_size)
            ent, _ = model.regularisers()
            print(f"step {step:5d} | soft BPC {soft:.3f} | hard BPC {hard:.3f} | entropy {ent.item():.3f} | {time.time()-t0:.0f}s")

        if step == args.steps:
            break
            
        # V6: Drop LR when STE starts to prevent shattering
        if step == args.hard_start:
            print(f"\n--- Step {step}: Starting Gentle STE phase (aligning readouts) ---")
            for pg in opt.param_groups:
                pg['lr'] *= 0.2

        x, y = get_batch(train, args.batch, args.ctx, args.device)

        # 1. Soft forward
        model.set_ste(False)
        logits_soft = model(x)
        ce_soft = F.cross_entropy(logits_soft.reshape(-1, 256), y.reshape(-1))

        # 2. Hard / STE forward
        hard_ce = torch.tensor(0.0, device=args.device)
        hard_w = 0.0
        if step >= args.hard_start and args.hard_w > 0:
            hard_w = args.hard_w * min(1.0, (step - args.hard_start + 1) / max(1, args.hard_ramp))
            model.set_ste(True)
            logits_hard = model(x)
            hard_ce = F.cross_entropy(logits_hard.reshape(-1, 256), y.reshape(-1))
            model.set_ste(False)

        # 3. Regularizers
        ent, perm_pen = model.regularisers()

        # 4. Final Loss
        lam = args.ent_max * min(1.0, (step + 1) / max(1, args.ent_warmup))
        loss = ce_soft + hard_w * hard_ce + lam * ent + args.perm_w * perm_pen

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
        opt.step()

    soft, hard = evaluate(model, all_val, args.device, chunk=args.eval_chunk_size)
    print(f"\nFINAL on full val ({all_val[:, 1:].numel()} chars): soft {soft:.3f} | HARD {hard:.3f} | bigram {bi:.3f} | trigram {tri:.3f}")
    torch.save(model.cpu().export_hard(), "gf256_hard_model.pt")
    print("saved hardened integer program -> gf256_hard_model.pt")

def build_parser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default=None)
    ap.add_argument("--d", type=int, default=32)
    ap.add_argument("--seed", type=int, default=32)
    ap.add_argument("--layers", type=int, default=3)
    ap.add_argument("--fan_in", type=int, default=3)
    ap.add_argument("--steps", type=int, default=12000)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--ctx", type=int, default=128)
    
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--clip", type=float, default=1.0)
    
    # V6 Hyperparams: Extreme entropy penalty to force 99.9% confidence!
    ap.add_argument("--ent_max", type=float, default=2.0)
    ap.add_argument("--ent_warmup", type=int, default=1000) # Warmup so it doesn't shatter at step 0
    ap.add_argument("--perm_w", type=float, default=1.0)
    
    # V6: Late gentle STE
    ap.add_argument("--hard_start", type=int, default=10000)
    ap.add_argument("--hard_ramp", type=int, default=2000)
    ap.add_argument("--hard_w", type=float, default=0.5)
    
    ap.add_argument("--eval_every", type=int, default=500)
    ap.add_argument("--eval_seqs", type=int, default=128)
    ap.add_argument("--eval_chunk_size", type=int, default=8)
    ap.add_argument("--no_ckpt", action="store_true")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    return ap

def run_main_with_args(args_list):
    main(build_parser().parse_args(args_list))

#if __name__ == "__main__":
#    main(build_parser().parse_args())
######################

s: {"date":"2026-09-30T20:58:26.628Z","msg_id":"909faf65-b0c5-453b-b765-fbef94a511fe","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#X22sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"run_main_with_args([\n    '--steps', '12000',\n    '--batch', '4',\n    '--seed', '32',\n    '--ctx', '128',\n    '--eval_every', '400',\n    '--eval_seqs', '64',\n    '--eval_chunk_size', '4',\n    '--no_ckpt'\n])"}


In [31]:
run_main_with_args([
    '--steps', '120000',
    '--batch', '4',
    '--seed', '32', ##golden seed
    '--ctx', '128',
    '--eval_every', '400',
    '--eval_seqs', '64',
    '--eval_chunk_size', '4',
    '--no_ckpt'
])

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cuda
step     0 | soft BPC 3.590 | hard BPC 3.588 | entropy 0.590 | 0s
step   400 | soft BPC 2.939 | hard BPC 4.112 | entropy 0.232 | 14s
step   800 | soft BPC 2.839 | hard BPC 3.980 | entropy 0.133 | 27s
step  1200 | soft BPC 2.804 | hard BPC 4.003 | entropy 0.095 | 40s
step  1600 | soft BPC 2.787 | hard BPC 4.023 | entropy 0.080 | 53s
step  2000 | soft BPC 2.756 | hard BPC 3.944 | entropy 0.072 | 66s
step  2400 | soft BPC 2.766 | hard BPC 3.914 | entropy 0.066 | 79s
step  2800 | soft BPC 2.775 | hard BPC 3.957 | entropy 0.061 | 92s
step  3200 | soft BPC 2.787 | hard BPC 3.997 | entropy 0.057 | 105s
step  3600 | soft BPC 2.784 | hard BPC 4.016 | entropy 0.054 | 119s
step  4000 | soft BPC 2.781 | hard BPC 4.087 | entropy 0.052 | 132s
step  4400 | soft BPC 2.762 | hard BPC 4.062 | entropy 0.049 | 145s
step  4800 | soft BPC 2.797 | hard BPC 4.089 | entropy 0.047 | 158s
step  5200 | s

s: {"date":"2026-09-30T23:42:03.958Z","msg_id":"0b8fbfce-3696-4e66-8866-d315789ff1ba","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#X24sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"import os\nfrom pathlib import Path\n\ncwd = Path.cwd()\nprint('cwd:', cwd)\nfor candidate in [cwd / 'gf256_hard_model.pt', Path.home() / 'gf256_hard_model.pt']:\n    print(candidate, 'exists=' + str(candidate.exists()))\n"}


In [28]:
run_main_with_args([
    '--steps', '12000',
    '--batch', '4',
    '--seed', '32', ##golden seed
    '--ctx', '128',
    '--eval_every', '400',
    '--eval_seqs', '64',
    '--eval_chunk_size', '4',
    '--no_ckpt'
])

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cuda
step     0 | soft BPC 3.590 | hard BPC 3.588 | entropy 0.590 | 0s
step   400 | soft BPC 2.939 | hard BPC 4.112 | entropy 0.232 | 14s
step   800 | soft BPC 2.839 | hard BPC 3.980 | entropy 0.133 | 27s
step  1200 | soft BPC 2.804 | hard BPC 4.003 | entropy 0.095 | 40s
step  1600 | soft BPC 2.787 | hard BPC 4.023 | entropy 0.080 | 53s
step  2000 | soft BPC 2.756 | hard BPC 3.944 | entropy 0.072 | 66s
step  2400 | soft BPC 2.766 | hard BPC 3.914 | entropy 0.066 | 79s
step  2800 | soft BPC 2.775 | hard BPC 3.957 | entropy 0.061 | 92s
step  3200 | soft BPC 2.787 | hard BPC 3.997 | entropy 0.057 | 105s
step  3600 | soft BPC 2.784 | hard BPC 4.016 | entropy 0.054 | 118s
step  4000 | soft BPC 2.781 | hard BPC 4.087 | entropy 0.052 | 132s
step  4400 | soft BPC 2.762 | hard BPC 4.062 | entropy 0.049 | 145s
step  4800 | soft BPC 2.797 | hard BPC 4.089 | entropy 0.047 | 158s
step  5200 | s

s: {"date":"2026-09-30T21:08:50.447Z","msg_id":"a0e70aff-b385-403e-8820-b0175463be34","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#X23sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"run_main_with_args([\n    '--steps', '12000',\n    '--batch', '4',\n    '--seed', '32', ##golden seed\n    '--ctx', '128',\n    '--eval_every', '400',\n    '--eval_seqs', '64',\n    '--eval_chunk_size', '4',\n    '--no_ckpt'\n])"}


In [15]:
run_main_with_args([
    '--steps', '12000',
    '--batch', '4',
    '--ctx', '128',
    '--eval_every', '500',
    '--eval_seqs', '64',
    '--eval_chunk_size', '4',
    '--no_ckpt'
])

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cuda
step     0 | soft BPC 3.590 | hard BPC 3.588 | entropy 0.590 nats | 0s
step   500 | soft BPC 2.867 | hard BPC 4.171 | entropy 0.240 nats | 17s
step  1000 | soft BPC 2.824 | hard BPC 3.992 | entropy 0.146 nats | 32s
step  1500 | soft BPC 2.796 | hard BPC 4.038 | entropy 0.116 nats | 49s
step  2000 | soft BPC 2.745 | hard BPC 4.097 | entropy 0.104 nats | 65s
step  2500 | soft BPC 2.721 | hard BPC 4.196 | entropy 0.097 nats | 81s
step  3000 | soft BPC 2.703 | hard BPC 4.258 | entropy 0.092 nats | 97s
step  3500 | soft BPC 2.720 | hard BPC 4.487 | entropy 0.088 nats | 113s
step  4000 | soft BPC 2.703 | hard BPC 4.551 | entropy 0.086 nats | 129s
step  4500 | soft BPC 2.693 | hard BPC 4.623 | entropy 0.083 nats | 145s
step  5000 | soft BPC 2.677 | hard BPC 4.695 | entropy 0.082 nats | 162s
step  5500 | soft BPC 2.698 | hard BPC 4.965 | entropy 0.080 nats | 178s
step  6000 | soft BPC

s: {"date":"2026-09-30T17:44:04.269Z","msg_id":"7493b905-2bb6-4f63-ab70-f2dd1ed8f2ba","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#X22sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"run_main_with_args([\n    '--steps', '12000',\n    '--batch', '4',\n    '--ctx', '128',\n    '--eval_every', '500',\n    '--eval_seqs', '64',\n    '--eval_chunk_size', '4',\n    '--no_ckpt'\n])"}


In [14]:
run_main_with_args([
    '--steps', '20000',
    '--batch', '4',
    '--ctx', '128',
    '--eval_every', '500',
    '--eval_seqs', '64',
    '--eval_chunk_size', '4',
    '--no_ckpt'
])

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cuda
step     0 | soft BPC 3.590 | hard BPC 3.588 | entropy 0.590 nats | 0s
step   500 | soft BPC 2.876 | hard BPC 4.148 | entropy 0.240 nats | 16s
step  1000 | soft BPC 2.801 | hard BPC 3.874 | entropy 0.146 nats | 32s
step  1500 | soft BPC 2.758 | hard BPC 4.044 | entropy 0.114 nats | 49s
step  2000 | soft BPC 2.718 | hard BPC 3.931 | entropy 0.101 nats | 66s
step  2500 | soft BPC 2.698 | hard BPC 3.977 | entropy 0.094 nats | 84s
step  3000 | soft BPC 2.676 | hard BPC 4.152 | entropy 0.089 nats | 102s
step  3500 | soft BPC 2.692 | hard BPC 4.261 | entropy 0.086 nats | 120s
step  4000 | soft BPC 2.689 | hard BPC 4.388 | entropy 0.083 nats | 138s
step  4500 | soft BPC 2.669 | hard BPC 4.453 | entropy 0.081 nats | 156s
step  5000 | soft BPC 2.666 | hard BPC 4.628 | entropy 0.080 nats | 174s
step  5500 | soft BPC 2.646 | hard BPC 4.676 | entropy 0.078 nats | 192s
step  6000 | soft BP

s: {"date":"2026-09-30T17:33:22.607Z","msg_id":"751f7356-ed3c-47cc-af13-a80a6bfbd6a1","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#X21sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"run_main_with_args([\n    '--steps', '12000',\n    '--batch', '4',\n    '--ctx', '128',\n    '--eval_every', '500',\n    '--eval_seqs', '64',\n    '--eval_chunk_size', '4',\n    '--no_ckpt'\n])"}


In [9]:
run_main_with_args([
    '--steps', '12000',
    '--batch', '4',
    '--ctx', '128',
    '--eval_every', '500',
    '--eval_seqs', '64',
    '--eval_chunk_size', '4',
    '--no_ckpt'
])

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cuda
step     0 | soft BPC 3.590 | hard BPC 3.588 | entropy 0.590 | 0s
step   500 | soft BPC 2.888 | hard BPC 4.113 | entropy 0.196 | 17s
step  1000 | soft BPC 2.834 | hard BPC 3.942 | entropy 0.110 | 36s
step  1500 | soft BPC 2.829 | hard BPC 4.143 | entropy 0.084 | 55s
step  2000 | soft BPC 2.793 | hard BPC 4.097 | entropy 0.073 | 74s
step  2500 | soft BPC 2.786 | hard BPC 4.149 | entropy 0.066 | 93s
step  3000 | soft BPC 2.797 | hard BPC 4.205 | entropy 0.062 | 111s
step  3500 | soft BPC 2.757 | hard BPC 4.244 | entropy 0.058 | 129s
step  4000 | soft BPC 2.773 | hard BPC 4.360 | entropy 0.055 | 148s
step  4500 | soft BPC 2.799 | hard BPC 4.254 | entropy 0.053 | 166s
step  5000 | soft BPC 2.754 | hard BPC 4.245 | entropy 0.050 | 185s
step  5500 | soft BPC 2.785 | hard BPC 4.143 | entropy 0.049 | 204s
step  6000 | soft BPC 2.800 | hard BPC 4.159 | entropy 0.047 | 223s
step  6500 |

s: {"date":"2026-09-30T15:28:33.763Z","msg_id":"6b22d2f2-5506-44ec-89aa-5a0a357c5f27","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#W0sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"\"\"\"gf256_markov_lm.py  -  Lagrange Field Transformer, prototype v4\n=================================================================\nV4 changes:\n  * Residual identity init from V3\n  * Hard-aware training:\n      - adds STE loss early\n      - soft CE and hard/STE CE are trained together\n  * Activation entropy penalty:\n      - penalizes multi-modal hidden distributions\n      - helps the exported argmax program match the soft model\n\"\"\"\nimport argparse, math, os, time, urllib.request\nimport torch\nimport torch.nn as nn\nimport torch.nn.function

In [4]:
print("cuda" if torch.cuda.is_available() else "cpu")

cuda


s: {"date":"2026-09-30T14:55:44.581Z","msg_id":"fdaf9396-be60-406f-8f2f-26b785e8ee73","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#X13sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"\"\"\"gf256_markov_lm.py  -  Lagrange Field Transformer, prototype v5\n=================================================================\nV5 changes:\n  * Returns to the V3 parameter-entropy minimization (which successfully \n    drove soft BPC down to 2.6).\n  * Removes the destructive weight decay and activation entropy from V4.\n  * Adds the hard/STE loss EARLY to prevent readouts from exploiting \n    soft softmax \"tails\", fixing the hard drift problem.\n\"\"\"\nimport argparse, math, os, time, urllib.request\nimport torch\nimport torch.nn as nn\nimp

In [6]:
run_main_with_args([
    '--steps', '12000',
    '--batch', '4',
    '--ctx', '128',
    '--eval_every', '500',
    '--eval_seqs', '64',
    '--eval_chunk_size', '4',
    '--no_ckpt',
    
    '--ent_max', '0.3',
    '--ent_warmup', '0',
    
    '--hard_start', '500',
    '--hard_ramp', '2000',
    '--hard_w', '1.0',
    
    '--lr', '3e-3'
])

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cuda
step     0 | soft BPC 3.590 | hard BPC 3.588 | entropy 0.590 | 1s
step   500 | soft BPC 2.883 | hard BPC 4.084 | entropy 0.256 | 17s
step  1000 | soft BPC 2.776 | hard BPC 3.520 | entropy 0.226 | 51s
step  1500 | soft BPC 2.822 | hard BPC 3.926 | entropy 0.326 | 82s
step  2000 | soft BPC 2.950 | hard BPC 4.067 | entropy 0.722 | 115s
step  2500 | soft BPC 2.988 | hard BPC 4.062 | entropy 0.954 | 146s
step  3000 | soft BPC 3.006 | hard BPC 3.700 | entropy 1.127 | 178s
step  3500 | soft BPC 3.020 | hard BPC 3.664 | entropy 1.286 | 214s
step  4000 | soft BPC 3.026 | hard BPC 3.815 | entropy 1.431 | 252s
step  4500 | soft BPC 3.029 | hard BPC 3.855 | entropy 1.543 | 284s
step  5000 | soft BPC 3.067 | hard BPC 3.889 | entropy 1.636 | 318s
step  5500 | soft BPC 3.211 | hard BPC 4.263 | entropy 1.737 | 353s
step  6000 | soft BPC 3.597 | hard BPC 4.226 | entropy 1.820 | 389s
step  6500

s: {"date":"2026-09-30T14:58:28.510Z","msg_id":"36b2dc5c-08f0-49aa-8d54-629160ccb9ef","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {}
s: {"silent":false,"store_history":false,"user_expressions":{},"allow_stdin":false,"stop_on_error":false,"code":"def __jupyter_exec_background__():\n    from IPython.display import display\n    from threading import Thread\n    from traceback import format_exc\n\n    # First send a dummy response to get the display id.\n    # Later we'll send the real response with the actual data.\n    # And that can happen much later even after the execution completes,\n    # as that response will be sent from a bg thread.\n    output = display({\"application/vnd.vscode.bg.execution.0\": \"\"}, raw=True, display_id=True)\n\n    def do_implementation():\n        # Copyright (c) Microsoft Corporation. All rights reserved.\n        # Licensed under the MIT License.

In [7]:
run_main_with_args([
    '--steps', '8000',
    '--batch', '4',
    '--ctx', '64',
    '--eval_every', '250',
    '--eval_seqs', '64',
    '--eval_chunk_size', '4',
    '--no_ckpt',
    '--hard_start', '500',
    '--hard_ramp', '1500',
    '--hard_w', '1.0',
    '--ent_max', '0.1',
    '--act_ent_w', '0.0',
    '--lr', '3e-3'
])

usage: ipykernel_launcher.py [-h] [--data DATA] [--d D] [--layers LAYERS]
                             [--fan_in FAN_IN] [--steps STEPS] [--batch BATCH]
                             [--ctx CTX] [--lr LR] [--clip CLIP]
                             [--ent_max ENT_MAX] [--ent_warmup ENT_WARMUP]
                             [--perm_w PERM_W] [--hard_start HARD_START]
                             [--hard_ramp HARD_RAMP] [--hard_w HARD_W]
                             [--eval_every EVAL_EVERY] [--eval_seqs EVAL_SEQS]
                             [--eval_chunk_size EVAL_CHUNK_SIZE] [--no_ckpt]
                             [--device DEVICE]
ipykernel_launcher.py: error: unrecognized arguments: --act_ent_w 0.0


SystemExit: 2

C:\Users\Chris4K\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3557: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


s: {"date":"2026-09-30T15:13:33.127Z","msg_id":"3571fb45-d46b-465d-835e-5e5194b9d937","msg_type":"execute_request","session":"5317e22e-e80d-4f1a-bf00-c2511daf8657","username":"adc20eed-2614-4888-9ec3-3ef87c5e06ab","version":"5.2"}
s: {}
s: {"cellId":"vscode-notebook-cell:Untitled-1.ipynb?jupyter-notebook#X13sdW50aXRsZWQ%3D"}
s: {"silent":false,"store_history":true,"user_expressions":{},"allow_stdin":true,"stop_on_error":false,"code":"\"\"\"gf256_markov_lm.py  -  Lagrange Field Transformer, prototype v6\n=================================================================\nV6 changes:\n  * CRANKS UP parameter entropy penalty (ent_max=2.0) to force 99.9% \n    deterministic S-boxes, preventing \"Cascading Chaos\" in the hard path.\n  * Delays STE loss until step 10000 to avoid noisy gradients on broad distributions.\n  * Drops LR by 5x when STE starts to gently align the readout tables.\n\"\"\"\nimport argparse, math, os, time, urllib.request\nimport torch\nimport torch.nn as nn\nimport tor

----


In [3]:
run_main_with_args([
    '--steps', '10000',
    '--batch', '8',
    '--ctx', '64',
    '--eval_chunk_size', '8'
])

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cpu


KeyboardInterrupt: 

In [7]:
"""gf256_markov_lm.py  -  Lagrange Field Transformer, prototype v4
=================================================================
V4 changes:
  * Residual identity init from V3
  * Hard-aware training:
      - adds STE loss early
      - soft CE and hard/STE CE are trained together
  * Activation entropy penalty:
      - penalizes multi-modal hidden distributions
      - helps the exported argmax program match the soft model
"""
import argparse, math, os, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

# ----------------------------------------------------------------------------
# GF(2^8) arithmetic: AES polynomial x^8+x^4+x^3+x+1, generator 3
# ----------------------------------------------------------------------------
def build_gf_tables():
    exp = [0] * 512
    log = [0] * 256
    x = 1
    for i in range(255):
        exp[i] = x
        log[x] = i

        # multiply x by generator 3 = x*2 XOR x
        x2 = x << 1
        if x2 & 0x100:
            x2 ^= 0x11B
        x = x2 ^ x

    for i in range(255, 512):
        exp[i] = exp[i - 255]
    return exp, log

EXP, LOG = build_gf_tables()

def gf_mul(a, b):
    if a == 0 or b == 0:
        return 0
    return EXP[LOG[a] + LOG[b]]

def gf_inv(a):
    return EXP[255 - LOG[a]]

# ----------------------------------------------------------------------------
# Walsh-Hadamard transform over last dim, size 256
# ----------------------------------------------------------------------------
def fwht(x):
    shape = x.shape
    x = x.reshape(-1, 256)
    h = 1
    while h < 256:
        x = x.view(-1, 256 // (2 * h), 2, h)
        a, b = x[:, :, 0, :], x[:, :, 1, :]
        x = torch.stack((a + b, a - b), dim=2)
        h *= 2
    return x.reshape(shape)

def xor_convolve(dists):
    """
    Distribution of XOR of independent GF(2^8)-valued variables.
    Each input tensor has last dimension 256 and sums to 1.
    """
    acc = None
    for d in dists:
        w = fwht(d)
        acc = w if acc is None else acc * w

    out = fwht(acc) / 256.0
    out = out.clamp_min(0.0)
    return out / out.sum(-1, keepdim=True).clamp_min(1e-12)

def ste_softmax(logits, ste):
    """
    If ste=False: normal soft probability distribution.
    If ste=True : hard one-hot forward, soft gradients backward.
    """
    soft = F.softmax(logits, -1)
    if not ste:
        return soft

    hard = F.one_hot(
        soft.argmax(dim=-1),
        num_classes=logits.shape[-1]
    ).to(logits.dtype)

    return hard + (soft - soft.detach())

def row_entropy(p):
    return -(p * (p + 1e-12).log()).sum(-1).mean()

# ----------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------
class GFLayer(nn.Module):
    """
    out_i = SBOX_i( XOR_f  c_{i,f} * x[channel_{i,f}, t - lag_{i,f}] )
    """

    def __init__(self, d, fan_in, lags, coef_set, gen):
        super().__init__()
        self.d, self.fan_in, self.lags = d, fan_in, lags

        K = len(coef_set)

        # Random wiring
        ch = torch.randint(0, d, (d, fan_in), generator=gen)
        lg = torch.randint(0, len(lags), (d, fan_in), generator=gen)

        # Path 0: own channel, lag 0
        ch[:, 0] = torch.arange(d)
        lg[:, 0] = lags.index(0)

        self.register_buffer("ch", ch)
        self.register_buffer("lag", lg)
        self.register_buffer("lag_vals", torch.tensor(lags)[lg])

        # Coefficient choices.
        # Residual init:
        #   path 0 -> multiply by 1
        #   paths >0 -> multiply by 0, disabled initially
        self.coef_logits = nn.Parameter(
            0.1 * torch.randn(d, fan_in, K, generator=gen)
        )

        with torch.no_grad():
            one_idx = coef_set.index(1)
            zero_idx = coef_set.index(0)

            # primary path strongly multiply-by-1
            self.coef_logits[:, 0, one_idx] += 6.0

            # other paths strongly multiply-by-0 initially
            if fan_in > 1:
                self.coef_logits[:, 1:, zero_idx] += 4.0

        # S-boxes: start near identity
        sbox = 0.1 * torch.randn(d, 256, 256, generator=gen)
        ar256 = torch.arange(256)
        sbox[
            torch.arange(d)[:, None],
            ar256[None, :],
            ar256[None, :]
        ] += 8.0

        self.sbox_logits = nn.Parameter(sbox)
        self.ste = False

    def sbox(self, ste=False):
        return ste_softmax(self.sbox_logits, ste)

    def coefs(self, ste=False):
        return ste_softmax(self.coef_logits, ste)

    def forward(self, x, perm_mats):
        # x: (B, T, d, 256)
        B, T, d, _ = x.shape
        max_lag = max(self.lags)

        pad = torch.zeros(
            B, max_lag, d, 256,
            device=x.device,
            dtype=x.dtype
        )
        pad[..., 0] = 1.0

        xp = torch.cat([pad, x], dim=1)  # (B, T+max_lag, d, 256)

        M = torch.einsum(
            "dfk,kij->dfij",
            self.coefs(self.ste),
            perm_mats
        )  # (d, F, 256, 256)

        t = torch.arange(T, device=x.device)[:, None]
        parts = []

        for f in range(self.fan_in):
            tidx = t + max_lag - self.lag_vals[:, f][None, :]  # (T,d)
            src = xp[:, tidx, self.ch[:, f][None, :]]          # (B,T,d,256)
            parts.append(
                torch.einsum("btdi,dij->btdj", src, M[:, f])
            )

        z = xor_convolve(parts)

        return torch.einsum(
            "btdi,dij->btdj",
            z,
            self.sbox(self.ste)
        )


class GFMarkovLM(nn.Module):
    def __init__(
        self,
        d=32,
        layers=2,
        fan_in=3,
        lags=(0, 1, 2, 4, 8, 16, 32, 64),
        n_coef=8,
        bigram_logp=None,
        seed=0
    ):
        super().__init__()
        gen = torch.Generator().manual_seed(seed)

        self.d = d
        self.lags = list(lags)

        # GF coefficient set
        coef_set = [0, 1]
        while len(coef_set) < n_coef:
            c = int(torch.randint(2, 256, (1,), generator=gen))
            if c not in coef_set:
                coef_set.append(c)
        self.coef_set = coef_set

        # Matrix for y = c*x over GF(2^8)
        pm = torch.zeros(len(coef_set), 256, 256)
        for k, c in enumerate(coef_set):
            for i in range(256):
                pm[k, i, gf_mul(c, i)] = 1.0
        self.register_buffer("perm_mats", pm)

        self.use_ckpt = True

        # Embedding:
        # channel 0 identity, other channels random bijections.
        emb = 0.1 * torch.randn(256, d, 256, generator=gen)
        ar256 = torch.arange(256)

        emb[ar256, 0, ar256] += 10.0

        for j in range(1, d):
            emb[ar256, j, torch.randperm(256, generator=gen)] += 8.0

        self.emb_logits = nn.Parameter(emb)

        self.layers = nn.ModuleList([
            GFLayer(d, fan_in, self.lags, coef_set, gen)
            for _ in range(layers)
        ])

        # Readout tables per stage: embedding + each layer
        self.readout = nn.ParameterList([
            nn.Parameter(torch.zeros(d, 256, 256))
            for _ in range(layers + 1)
        ])

        # Markov init
        if bigram_logp is not None:
            with torch.no_grad():
                self.readout[0][0] = bigram_logp

        self.readout_bias = nn.Parameter(torch.zeros(256))
        self.ste = False

        # For activation entropy logging/training
        self.last_act_ent = 0.0

    def set_ste(self, on):
        self.ste = on
        for L in self.layers:
            L.ste = on

    def forward(self, tokens, track_act_ent=False):
        # h: (B,T,d,256)
        h = ste_softmax(self.emb_logits, self.ste)[tokens]

        if track_act_ent:
            act_ent = row_entropy(h)
            cnt = 1.0
        else:
            act_ent = 0.0
            cnt = 0.0

        logits = torch.einsum(
            "btdi,div->btv",
            h,
            self.readout[0]
        )

        for layer, R in zip(self.layers, self.readout[1:]):
            if self.training and self.use_ckpt:
                h = torch.utils.checkpoint.checkpoint(
                    layer,
                    h,
                    self.perm_mats,
                    use_reentrant=False
                )
            else:
                h = layer(h, self.perm_mats)

            if track_act_ent:
                act_ent = act_ent + row_entropy(h)
                cnt += 1.0

        if track_act_ent:
            self.last_act_ent = act_ent / max(1.0, cnt)
        else:
            self.last_act_ent = 0.0

        return logits + self.readout_bias

    def regularisers(self):
        ent = row_entropy(F.softmax(self.emb_logits, -1))
        perm_pen = 0.0

        for L in self.layers:
            # Always compute regularizers on the soft distribution
            P = L.sbox(False)
            C = L.coefs(False)

            ent = ent + row_entropy(P) + row_entropy(C)
            perm_pen = perm_pen + ((P.sum(1) - 1.0) ** 2).mean()

        ent = ent / (1 + 2 * len(self.layers))
        perm_pen = perm_pen / max(1, len(self.layers))

        return ent, perm_pen

    # ------------------------------------------------------------------------
    # Hardened model: pure integer program
    # ------------------------------------------------------------------------
    @torch.no_grad()
    def export_hard(self):
        return {
            "emb": self.emb_logits.argmax(-1),
            "coef": [
                torch.tensor(
                    self.coef_set,
                    device=L.coef_logits.device
                )[L.coef_logits.argmax(-1)]
                for L in self.layers
            ],
            "sbox": [L.sbox_logits.argmax(-1) for L in self.layers],
            "wiring": [(L.ch, L.lag) for L in self.layers],
            "readout": [R.detach() for R in self.readout],
            "bias": self.readout_bias.detach(),
        }

    @torch.no_grad()
    def hard_forward(self, tokens, hard):
        exp_t = torch.tensor(EXP)
        log_t = torch.tensor(LOG)
        lags = torch.tensor(self.lags)

        B, T = tokens.shape
        d = self.d

        h = hard["emb"][tokens]  # (B,T,d)
        ar = torch.arange(d)

        logits = hard["readout"][0][ar, h].sum(2)

        for coef, sbox, (ch, lg), R in zip(
            hard["coef"],
            hard["sbox"],
            hard["wiring"],
            hard["readout"][1:]
        ):
            acc = torch.zeros(B, T, d, dtype=torch.long)

            for f in range(coef.shape[1]):
                out = torch.zeros(B, T, d, dtype=torch.long)

                for j in range(d):
                    l = int(lags[lg[j, f]])
                    src = h[:, :, ch[j, f]]

                    if l:
                        src = torch.cat([
                            torch.zeros(B, l, dtype=torch.long),
                            src[:, :-l]
                        ], dim=1)

                    c = int(coef[j, f])

                    if c == 0:
                        prod = torch.zeros_like(src)
                    else:
                        prod = torch.where(
                            src == 0,
                            torch.zeros_like(src),
                            exp_t[log_t[src] + LOG[c]]
                        )

                    out[:, :, j] = prod

                acc = acc ^ out

            h = sbox[ar, acc]
            logits = logits + R[ar, h].sum(2)

        return logits + hard["bias"]


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

def load_data(path):
    if path is None:
        path = "tinyshakespeare.txt"
        if not os.path.exists(path):
            print("downloading Tiny Shakespeare ...")
            urllib.request.urlretrieve(URL, path)

    raw = open(path, "rb").read()
    data = torch.tensor(list(raw), dtype=torch.long)

    n = int(0.9 * len(data))
    return data[:n], data[n:]

def bigram_logp(data):
    counts = torch.ones(256, 256)  # Laplace smoothing
    counts.index_put_(
        (data[:-1], data[1:]),
        torch.ones(len(data) - 1),
        accumulate=True
    )
    return (counts / counts.sum(1, keepdim=True)).log()

def ngram_baselines(train, val):
    uni = torch.bincount(train, minlength=256).float() + 1
    uni = uni / uni.sum()

    c2 = torch.bincount(
        train[:-1] * 256 + train[1:],
        minlength=256**2
    ).float().view(256, 256)

    c3 = torch.bincount(
        train[:-2] * 65536 + train[1:-1] * 256 + train[2:],
        minlength=256**3
    ).float()

    a, b, c = val[:-2], val[1:-1], val[2:]

    p1 = uni[c]
    p2 = c2[b, c] / c2[b].sum(1).clamp_min(1)

    ab = c2[a, b]
    p3 = c3[a * 65536 + b * 256 + c] / ab.clamp_min(1)

    bi = -(0.95 * p2 + 0.05 * p1).log2().mean().item()
    tri = -(0.6 * p3 + 0.3 * p2 + 0.1 * p1).log2().mean().item()

    return bi, tri

def get_batch(data, B, T, device):
    ix = torch.randint(0, len(data) - T - 1, (B,))
    x = torch.stack([data[i:i + T] for i in ix])
    y = torch.stack([data[i + 1:i + T + 1] for i in ix])
    return x.to(device), y.to(device)

def bpc(logits, y):
    return F.cross_entropy(
        logits.reshape(-1, 256),
        y.reshape(-1)
    ).item() / math.log(2)

@torch.no_grad()
def evaluate(model, seqs, device, chunk=64):
    """
    Returns soft BPC and hard BPC.
    """
    was_ste = model.ste
    model.eval()
    model.set_ste(False)

    x, y = seqs[:, :-1], seqs[:, 1:]
    soft_nll = 0.0
    hard_nll = 0.0

    for k in range(0, len(x), chunk):
        xb = x[k:k + chunk]
        yb = y[k:k + chunk]

        lo = model(xb.to(device)).float().cpu()
        soft_nll += F.cross_entropy(
            lo.reshape(-1, 256),
            yb.reshape(-1),
            reduction="sum"
        ).item()

    hard = model.export_hard()
    hard = {
        k: (
            [t.cpu() for t in v]
            if isinstance(v, list) and not isinstance(v[0], tuple)
            else [(a.cpu(), b.cpu()) for a, b in v]
            if isinstance(v, list)
            else v.cpu()
        )
        for k, v in hard.items()
    }

    for k in range(0, len(x), chunk):
        xb = x[k:k + chunk]
        yb = y[k:k + chunk]

        lo = model.hard_forward(xb, hard)
        hard_nll += F.cross_entropy(
            lo.reshape(-1, 256),
            yb.reshape(-1),
            reduction="sum"
        ).item()

    n = y.numel() * math.log(2)

    model.set_ste(was_ste)
    model.train()

    return soft_nll / n, hard_nll / n


# ----------------------------------------------------------------------------
# Training
# ----------------------------------------------------------------------------
def main(args):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    train, val = load_data(args.data)
    bi, tri = ngram_baselines(train, val)
    print(f"baselines on full val: bigram {bi:.3f} BPC | trigram {tri:.3f} BPC")

    model = GFMarkovLM(
        d=args.d,
        layers=args.layers,
        fan_in=args.fan_in,
        bigram_logp=bigram_logp(train)
    ).to(args.device)

    model.use_ckpt = not args.no_ckpt

    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    n_par = sum(p.numel() for p in model.parameters())
    print(
        f"params {n_par/1e6:.2f}M | d={args.d} layers={args.layers} "
        f"fan_in={args.fan_in} device={args.device}"
    )

    L = args.ctx + 1
    all_val = val[: (len(val) // L) * L].view(-1, L)

    gsel = torch.Generator().manual_seed(123)
    quick_val = all_val[
        torch.randperm(len(all_val), generator=gsel)[:args.eval_seqs]
    ]

    ent_warmup = args.ent_warmup if args.ent_warmup > 0 else 1000

    t0 = time.time()
    last_act_for_print = 0.0

    for step in range(args.steps + 1):
        if step % args.eval_every == 0 or step == args.steps:
            soft, hard = evaluate(
                model,
                quick_val,
                args.device,
                chunk=args.eval_chunk_size
            )
            ent, _ = model.regularisers()

            print(
                f"step {step:5d} | soft BPC {soft:.3f} | hard BPC {hard:.3f} "
                f"| entropy {ent.item():.3f} | act {last_act_for_print:.3f} "
                f"| {time.time()-t0:.0f}s"
            )

        if step == args.steps:
            break

        x, y = get_batch(train, args.batch, args.ctx, args.device)

        # ---------------------------------------------------------
        # Soft loss
        # ---------------------------------------------------------
        model.set_ste(False)
        logits_soft = model(
            x,
            track_act_ent=(args.act_ent_w > 0)
        )

        ce_soft = F.cross_entropy(
            logits_soft.reshape(-1, 256),
            y.reshape(-1)
        )

        act_ent_soft = model.last_act_ent
        if torch.is_tensor(act_ent_soft):
            last_act_for_print = act_ent_soft.item()
        else:
            last_act_for_print = float(act_ent_soft)

        # ---------------------------------------------------------
        # Hard / STE loss, started early
        # ---------------------------------------------------------
        hard_ce = torch.tensor(0.0, device=args.device)
        hard_w = 0.0

        if step >= args.hard_start and args.hard_w > 0:
            hard_w = args.hard_w * min(
                1.0,
                (step - args.hard_start + 1) / max(1, args.hard_ramp)
            )

            model.set_ste(True)
            logits_hard = model(x, track_act_ent=False)
            hard_ce = F.cross_entropy(
                logits_hard.reshape(-1, 256),
                y.reshape(-1)
            )
            model.set_ste(False)

        # ---------------------------------------------------------
        # Regularizers
        # ---------------------------------------------------------
        ent, perm_pen = model.regularisers()

        lam = args.ent_max * min(
            1.0,
            (step + 1) / max(1, ent_warmup)
        )

        act_w = args.act_ent_w * min(
            1.0,
            (step + 1) / max(1, ent_warmup)
        )

        loss = (
            ce_soft
            + hard_w * hard_ce
            + lam * ent
            + args.perm_w * perm_pen
            + act_w * act_ent_soft
        )

        opt.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)

        opt.step()

    soft, hard = evaluate(
        model,
        all_val,
        args.device,
        chunk=args.eval_chunk_size
    )

    print(
        f"\nFINAL on full val ({all_val[:, 1:].numel()} chars): "
        f"soft {soft:.3f} | HARD {hard:.3f} | "
        f"bigram {bi:.3f} | trigram {tri:.3f}"
    )

    torch.save(model.cpu().export_hard(), "gf256_hard_model.pt")
    print("saved hardened integer program -> gf256_hard_model.pt")


def build_parser():
    ap = argparse.ArgumentParser()

    ap.add_argument("--data", default=None)
    ap.add_argument("--d", type=int, default=32)
    ap.add_argument("--layers", type=int, default=3)
    ap.add_argument("--fan_in", type=int, default=3)
    ap.add_argument("--steps", type=int, default=10000)

    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--ctx", type=int, default=128)

    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--clip", type=float, default=1.0)

    # Parameter-row entropy penalty
    ap.add_argument("--ent_max", type=float, default=0.3)
    ap.add_argument("--ent_warmup", type=int, default=1000)
    ap.add_argument("--perm_w", type=float, default=1.0)

    # Activation entropy penalty
    ap.add_argument("--act_ent_w", type=float, default=0.3)

    # Hard-aware STE loss
    ap.add_argument("--hard_start", type=int, default=100)
    ap.add_argument("--hard_ramp", type=int, default=1000)
    ap.add_argument("--hard_w", type=float, default=1.0)

    ap.add_argument("--eval_every", type=int, default=250)
    ap.add_argument("--eval_seqs", type=int, default=128)
    ap.add_argument("--eval_chunk_size", type=int, default=8)

    ap.add_argument("--no_ckpt", action="store_true")
    ap.add_argument(
        "--device",
        default="cuda" if torch.cuda.is_available() else "cpu"
    )

    return ap

def run_main_with_args(args_list):
    """
    Example:
      run_main_with_args(['--steps', '10000', '--ctx', '128'])
    """
    main(build_parser().parse_args(args_list))
#
#if __name__ == "__main__":
#    main(build_parser().parse_args())

In [10]:
run_main_with_args([
    '--steps', '8000',
    '--batch', '4',
    '--ctx', '64',
    '--eval_every', '250',
    '--eval_seqs', '64',
    '--eval_chunk_size', '4',
    '--no_ckpt',

    '--hard_start', '2000',
    '--hard_ramp', '3000',
    '--hard_w', '0.3',

    '--act_ent_w', '0.0',
    '--ent_max', '0.0',

    '--lr', '3e-3'
])

baselines on full val: bigram 3.602 BPC | trigram 3.087 BPC
params 16.78M | d=32 layers=3 fan_in=3 device=cpu
step     0 | soft BPC 3.624 | hard BPC 3.623 | entropy 0.590 | act 0.000 | 2s
step   250 | soft BPC 3.689 | hard BPC 3.709 | entropy 0.597 | act 0.000 | 38s
step   500 | soft BPC 3.656 | hard BPC 3.675 | entropy 0.605 | act 0.000 | 73s
step   750 | soft BPC 3.666 | hard BPC 3.692 | entropy 0.613 | act 0.000 | 108s
step  1000 | soft BPC 3.650 | hard BPC 3.681 | entropy 0.622 | act 0.000 | 144s
step  1250 | soft BPC 3.654 | hard BPC 3.697 | entropy 0.631 | act 0.000 | 179s
step  1500 | soft BPC 3.658 | hard BPC 3.701 | entropy 0.640 | act 0.000 | 215s
step  1750 | soft BPC 3.643 | hard BPC 3.693 | entropy 0.650 | act 0.000 | 251s
step  2000 | soft BPC 3.641 | hard BPC 3.704 | entropy 0.659 | act 0.000 | 290s


KeyboardInterrupt: 

In [32]:
import os
from pathlib import Path

cwd = Path.cwd()
print('cwd:', cwd)
for candidate in [cwd / 'gf256_hard_model.pt', Path.home() / 'gf256_hard_model.pt']:
    print(candidate, 'exists=' + str(candidate.exists()))


cwd: c:\Users\Chris4K\AppData\Local\Programs\Microsoft VS Code
c:\Users\Chris4K\AppData\Local\Programs\Microsoft VS Code\gf256_hard_model.pt exists=True
C:\Users\Chris4K\gf256_hard_model.pt exists=False
